# srforge Demo

用合成数据演示完整流程：PySR 多轮训练 → 结构分析 → 公式锻造。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. 生成合成数据

真实公式：`y = 3.5 * sqrt(x0 * x1) + 2.1 * x2 + x3 + 5.0`

In [ ]:
np.random.seed(42)
n = 200
X = pd.DataFrame({
    'x0': np.random.uniform(1, 10, n),
    'x1': np.random.uniform(1, 10, n),
    'x2': np.random.uniform(0, 5, n),
    'x3': np.random.uniform(0, 5, n),
    'noise1': np.random.normal(0, 1, n),   # 噪声特征
    'noise2': np.random.normal(0, 1, n),
})
y = 3.5 * np.sqrt(X['x0'] * X['x1']) + 2.1 * X['x2'] + X['x3'] + 5.0 + np.random.normal(0, 0.5, n)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Train: {len(x_train)}, Test: {len(x_test)}')

## 2. 多轮 PySR 训练

In [ ]:
from pysr import PySRRegressor

equations_list = []
for seed in range(10):
    model = PySRRegressor(
        binary_operators=["+", "-", "*", "/", "pow"],
        unary_operators=["sqrt"],
        niterations=200,
        populations=8,
        population_size=30,
        maxsize=15,
        random_state=seed,
        deterministic=True,
        parallelism="serial",
        verbosity=0,
    )
    model.fit(x_train, y_train)
    equations_list.append(model.equations_)

print(f'训练完成，{len(equations_list)} 轮')

## 3. 一键分析

In [ ]:
from srforge import build_formulas, full_analysis

formulas = build_formulas(equations_list)
result = full_analysis(formulas)
# 终端打印 Pattern 报告 + 公式排名 + 浏览器打开 HTML 报告

## 4. 提取共识子式

In [ ]:
from srforge import extract_candidates, print_candidates

candidates = extract_candidates(formulas, result["scored"],
                                 result["patterns"], result["context"],
                                 x_filter=x_train)
print_candidates(candidates, topk=10)

## 5. ElasticNet 锻造公式

In [ ]:
from srforge import elastic_select, print_selection

sel = elastic_select(candidates, x_train, y_train, x_test, y_test)
print_selection(sel)
# 输出最终公式 + 测试集 R²/MSE

## 6. 稳定性验证

In [ ]:
from srforge import cross_validate, print_cv, check_formula

df = cross_validate(sel["formula"], X, y)
print_cv(df)

# 安全检查
warnings = check_formula(sel["formula"], X)
if warnings:
    for w in warnings:
        print(f"风险:{w}")
else:
    print("公式安全")

## 7. PySR Round 2（二次搜索）

可选：将共识子式喂回第二轮搜索

In [ ]:
# from srforge import round2_forge, expand_feats
# 
# # 需要定义 train_fn 函数
# r2 = round2_forge(result, formulas, x_train, y_train, x_test, y_test, train_fn=your_train_fn)
# 
# # 展开 feat_N → 原始表达式
# top_eq = r2["scored"][0]["equation"]
# eq = expand_feats(top_eq, r2["feat_map"])
# print(eq)